In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
import pandas as pd


In [3]:
df = pd.read_csv("../data/processed/demand_features.csv", parse_dates=["date"])
df.head()

,date,sku_id,units_sold,year,month,day,day_of_week,day_of_year,is_weekend,is_back_to_school,is_holiday_season,lag_1,lag_7,lag_14,rolling_mean_7,rolling_mean_30
0,2022-01-31,MBA13,234,2022,1,31,0,31,0,0,0,188.0,209.0,221.0,221.714286,227.966667
1,2022-02-01,MBA13,309,2022,2,1,1,32,0,0,0,234.0,236.0,262.0,225.285714,228.900000
2,2022-02-02,MBA13,252,2022,2,2,2,33,0,0,0,309.0,256.0,225.0,235.714286,232.833333
3,2022-02-03,MBA13,221,2022,2,3,3,34,0,0,0,252.0,218.0,209.0,235.142857,232.200000
4,2022-02-04,MBA13,278,2022,2,4,4,35,0,0,0,221.0,264.0,297.0,235.571429,229.633333


In [ ]:
feature_cols = [col for col in df.columns if col not in ["date", "sku_id", "units_sold"]]
# print("Feature columns:", feature_cols)
# models to compare
model_configs = {
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=0.1),
    "ElasticNet": ElasticNet(alpha=0.1, l1_ratio=0.5),
}

all_results = []
trained_models = {}  # trained_models[model_name][sku_id] = model object

for model_name, model_template in model_configs.items():
    trained_models[model_name] = {}
    
    for sku in df["sku_id"].unique():
        sku_df = df[df["sku_id"] == sku].sort_values("date").reset_index(drop=True)
        
        split_idx = int(len(sku_df) * 0.8)
        train = sku_df.iloc[:split_idx]
        test = sku_df.iloc[split_idx:]
        
        X_train, y_train = train[feature_cols], train["units_sold"]
        X_test, y_test = test[feature_cols], test["units_sold"]
        
        model = model_template.__class__(**model_template.get_params())  # fresh copy
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        
        rmse = np.sqrt(mean_squared_error(y_test, preds))
        mae = mean_absolute_error(y_test, preds)
        mape = np.mean(np.abs((y_test - preds) / y_test)) * 100
        r2 = r2_score(y_test, preds)
        
        all_results.append({
            "model": model_name,
            "sku_id": sku,
            "RMSE": rmse,
            "MAE": mae,
            "MAPE": mape,
            "R2": r2
        })
        trained_models[model_name][sku] = model

results_df = pd.DataFrame(all_results)
results_df

Feature columns: ['year', 'month', 'day', 'day_of_week', 'day_of_year', 'is_weekend', 'is_back_to_school', 'is_holiday_season', 'lag_1', 'lag_7', 'lag_14', 'rolling_mean_7', 'rolling_mean_30']


,model,sku_id,RMSE,MAE,MAPE,R2
0,LinearRegression,MBA13,42.479059,33.474336,10.746548,0.643648
1,LinearRegression,MBA15,41.657805,31.859473,11.004682,0.598112
2,LinearRegression,MBP14,30.888948,24.121067,11.647506,0.551182
3,LinearRegression,MBP16,16.338674,13.190738,11.022970,0.581642
4,Ridge,MBA13,42.533577,33.593019,10.822108,0.642733
5,Ridge,MBA15,41.630776,31.850260,11.036942,0.598633
6,Ridge,MBP14,30.884853,24.164381,11.705146,0.551301
7,Ridge,MBP16,16.327088,13.168849,11.021399,0.582235
8,Lasso,MBA13,42.555041,33.710871,10.870878,0.642373
9,Lasso,MBA15,41.641710,31.928533,11.079495,0.598422


In [5]:
summary_df = results_df.groupby("model")[["RMSE", "MAE", "MAPE", "R2"]].mean().round(3)
summary_df = summary_df.sort_values("RMSE")
summary_df

,RMSE,MAE,MAPE,R2
model,,,,
LinearRegression,32.841,25.661,11.105,0.594
Ridge,32.844,25.694,11.146,0.594
Lasso,32.863,25.762,11.184,0.593
ElasticNet,33.827,26.733,11.599,0.569


In [6]:
results_df.to_csv("../data/processed/baseline_regression_results.csv", index=False)
summary_df.to_csv("../data/processed/baseline_regression_summary.csv")
print("Saved baseline results.")

Saved baseline results.


In [8]:
print(df.groupby('sku_id')['units_sold'].mean())

sku_id
MBA13    289.180992
MBA15    245.270169
MBP14    185.429644
MBP16    110.936210
Name: units_sold, dtype: float64
